<a href="https://colab.research.google.com/github/sethspacedigitalconsulting-sketch/space-123/blob/main/04_real_llm_drift_rollback_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: Detect Prompt Regression and Roll Back

## Session outcome

Participants will test two prompt versions with a real open-source LLM, detect whether the new prompt regresses quality, and produce a rollback/postmortem note.

In [ ]:
import importlib.util
import json
import re
import statistics
import subprocess
import sys
import time
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from typing import Any, Dict, List


def install_if_missing(package: str, import_name: str | None = None) -> None:
    import_name = import_name or package.split(">=")[0].replace("-", "_")
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


install_if_missing("transformers>=4.44.0", "transformers")
install_if_missing("accelerate>=0.33.0", "accelerate")
install_if_missing("torch", "torch")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline


MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"


def load_llm(model_id: str = MODEL_ID):
    print(f"Loading open-source model: {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype)
    device = 0 if torch.cuda.is_available() else -1
    generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=device)
    print("Device:", "GPU" if torch.cuda.is_available() else "CPU")
    return tokenizer, generator


tokenizer, generator = load_llm()


def ask_llm(system: str, user: str, max_new_tokens: int = 180, temperature: float = 0.7) -> str:
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    try:
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        prompt = f"System: {system}\nUser: {user}\nAssistant:"

    output = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.9,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id,
    )[0]["generated_text"]
    return output.strip()


def extract_json(text: str) -> Dict[str, Any]:
    match = re.search(r"\{.*\}", text, flags=re.S)
    if not match:
        raise ValueError("No JSON object found in model output.")
    return json.loads(match.group(0))

Loading open-source model: HuggingFaceTB/SmolLM2-135M-Instruct


config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Device: GPU


## 1. Define Prompt Versions

In [ ]:
PROMPTS = {
    "assistant@v1.0.0": """You are an AI-First SDLC instructor.
Answer in 70-100 words. Include production details such as evals, monitoring, rollback, safety, latency, cost, or quality.
""",
    "assistant@v1.1.0": """You are a very brief assistant.
Answer in one or two sentences.
""",
}

active_prompt = "assistant@v1.0.0"
candidate_prompt = "assistant@v1.1.0"

## 2. Define Canary Questions

In [ ]:
canary_questions = [
    "How do evals help prevent bad AI releases?",
    "What should an AI operations dashboard track?",
    "Why is rollback important for prompt changes?",
    "How can a team respond to hallucination spikes?",
    "What should be checked before giving an AI agent tool access?",
]

## 3. Run the Canary

In [ ]:
def run_prompt_version(prompt_version: str) -> List[Dict[str, Any]]:
    rows = []
    for question in canary_questions:
        started = time.time()
        answer = ask_llm(PROMPTS[prompt_version], question, max_new_tokens=160, temperature=0.8)
        latency_ms = int((time.time() - started) * 1000)
        rows.append({
            "prompt_version": prompt_version,
            "question": question,
            "answer": answer,
            "latency_ms": latency_ms,
            "word_count": len(answer.split()),
            "mentions_controls": bool(re.search(r"eval|monitor|rollback|safety|human|cost|latency|quality|version", answer, re.I)),
        })
    return rows


active_results = run_prompt_version(active_prompt)
candidate_results = run_prompt_version(candidate_prompt)

candidate_results[0]

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'temperature', 'top_p', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[tra

{'prompt_version': 'assistant@v1.1.0',
 'question': 'How do evals help prevent bad AI releases?',
 'answer': 'Evals can help prevent bad AI releases by ensuring that the AI system is not malicious or intentionally designed to cause harm. By requiring a thorough review and verification process, we can catch any potential issues or flaws before releasing a product, which can prevent the creation of rogue or malicious AI.',
 'latency_ms': 2371,
 'word_count': 52,
 'mentions_controls': True}

## 4. Score Canary Results

In [ ]:
def summarize_canary(rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    return {
        "prompt_version": rows[0]["prompt_version"],
        "mean_word_count": round(statistics.mean(row["word_count"] for row in rows), 1),
        "mean_latency_ms": round(statistics.mean(row["latency_ms"] for row in rows), 1),
        "control_mention_rate": round(sum(row["mentions_controls"] for row in rows) / len(rows), 2),
    }


active_summary = summarize_canary(active_results)
candidate_summary = summarize_canary(candidate_results)

active_summary, candidate_summary

({'prompt_version': 'assistant@v1.0.0',
  'mean_word_count': 117.8,
  'mean_latency_ms': 6411.8,
  'control_mention_rate': 0.8},
 {'prompt_version': 'assistant@v1.1.0',
  'mean_word_count': 65.6,
  'mean_latency_ms': 3455.6,
  'control_mention_rate': 0.4})

## 5. Decide Promote or Roll Back

In [ ]:
ROLLBACK_RULES = {
    "min_control_mention_rate": 0.80,
    "min_mean_word_count": 35,
}


def decide(active: Dict[str, Any], candidate: Dict[str, Any]) -> Dict[str, Any]:
    reasons = []
    if candidate["control_mention_rate"] < ROLLBACK_RULES["min_control_mention_rate"]:
        reasons.append("candidate omits too many production control concepts")
    if candidate["mean_word_count"] < ROLLBACK_RULES["min_mean_word_count"]:
        reasons.append("candidate answers are too thin for a training context")
    if reasons:
        return {"decision": "ROLL BACK", "reasons": reasons}
    return {"decision": "PROMOTE", "reasons": ["candidate passed canary rules"]}


decision = decide(active_summary, candidate_summary)
decision

{'decision': 'ROLL BACK',
 'reasons': ['candidate omits too many production control concepts']}

## 6. Rollback and Postmortem Note

In [ ]:
deployment_state = {
    "current_prompt": candidate_prompt,
    "previous_prompt": active_prompt,
    "traffic_percentage": 25,
}

if decision["decision"] == "ROLL BACK":
    deployment_state["current_prompt"] = active_prompt
    deployment_state["traffic_percentage"] = 0

postmortem = {
    "incident": "Candidate prompt failed canary review",
    "detected_by": "Notebook canary eval",
    "decision": decision["decision"],
    "reasons": decision["reasons"],
    "resolution": f"Current prompt set to {deployment_state['current_prompt']}",
    "prevention": [
        "Add minimum operational-detail eval before prompt release.",
        "Run canary comparison before full rollout.",
        "Keep previous prompt version available for rollback.",
    ],
}

deployment_state, postmortem

({'current_prompt': 'assistant@v1.0.0',
  'previous_prompt': 'assistant@v1.0.0',
  'traffic_percentage': 0},
 {'incident': 'Candidate prompt failed canary review',
  'detected_by': 'Notebook canary eval',
  'decision': 'ROLL BACK',
  'reasons': ['candidate omits too many production control concepts'],
  'resolution': 'Current prompt set to assistant@v1.0.0',
  'prevention': ['Add minimum operational-detail eval before prompt release.',
   'Run canary comparison before full rollout.',
   'Keep previous prompt version available for rollback.']})

## Hands-on deliverable

Improve `assistant@v1.1.0` so that it stays concise but includes production controls. Rerun the canary and decide whether it can be promoted.